In [1]:
import torch
from lerobot.common.pointnet2_models import *
import importlib, sys
sys.modules['pointnet2_models'] = importlib.import_module('lerobot.common.pointnet2_models')
model = importlib.import_module("lerobot.common.pointnet2_models.pointnet2_cls_ssg")
import torch
import torch.nn.functional as F  # noqa: N812
import torchvision
from torch import Tensor, nn
from torchvision.models._utils import IntermediateLayerGetter
from torchvision.ops.misc import FrozenBatchNorm2d

In [2]:
import torch, importlib, re
import torch.nn as nn

CKPT = "Pointnet_Pointnet2_pytorch/log/classification/pointnet2_ssg_wo_normals/checkpoints/best_model.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class PN2SSGEncoder(nn.Module):
    def __init__(self, ckpt_path):
        super().__init__()
        m = importlib.import_module("lerobot.common.pointnet2_models.pointnet2_cls_ssg")
        self.net = m.get_model(num_class=40, normal_channel=False)

        raw = torch.load(ckpt_path, weights_only=False)
        sd  = raw.get("model_state_dict", raw.get("state_dict", raw))
        sd  = {k.replace("module.", ""): v for k, v in sd.items()}
        self.net.load_state_dict(sd, strict=True)
        self.net.to(DEVICE).eval()

    @torch.no_grad()
    def forward(self, xyz):              # xyz: (B, N, 3) or (B, 3, N)
        # Convert to (B, 3, N) for this implementation
        if xyz.shape[1] != 3 and xyz.shape[-1] == 3:
            xyz = xyz.permute(0, 2, 1).contiguous()
        xyz = xyz.to(DEVICE)

        logits, l3_points = self.net(xyz)     # l3_points: (B, 1024, 1)
        feats = l3_points.squeeze(-1)         # -> (B, 1024)
        return feats

encoder = PN2SSGEncoder(CKPT)
xyz = torch.randn(8, 4096, 3)                  # (B,N,3)
feat = encoder(xyz)
print(feat.shape)  # torch.Size([B, 1024])

torch.Size([8, 1024])


In [5]:
backbone_model = getattr(torchvision.models, "resnet18")(
                    replace_stride_with_dilation=[False, False, False],
                    weights="ResNet18_Weights.IMAGENET1K_V1",
                    norm_layer=FrozenBatchNorm2d,
                )
                # Note: The assumption here is that we are using a ResNet model (and hence layer4 is the final
                # feature map).
                # Note: The forward method of this returns a dict: {"feature_map": output}.
backbone = IntermediateLayerGetter(backbone_model, return_layers={"layer4": "feature_map"})
img = torch.randn(8, 3, 224, 224)  # Example input image tensor
cam_features = backbone(img)["feature_map"]
print(cam_features.shape)  # Should print the shape of the feature map from layer4
backbone_model.fc.in_features

torch.Size([8, 512, 7, 7])


512